## Regression tests for satellite-in-situ-matchup

This notebook runs regression tests against `satellite-in-situ-matchup`, the
Satellite–In Situ Matchup Service (SIMS) from the Earthdata Archive. It reads
Earthdata Archive Level-2 granules and, from version 1.1.0, PRISM airborne
granules from ORNL DAAC and GHRSST L2P granules from PO.DAAC.

A request carries one in-situ observation: a bounding box and a time range
centered on it, with `format=text/csv`. For each granule Harmony finds, the
service takes the 5×5 pixel box around the observation, applies the quality
rules for that kind of granule (for Earthdata Archive granules, those of
`mk_matchup.py` from seabass-utils), and returns one CSV row with a status
(`matched`, `rejected` or `not_in_granule`) and the pixel statistics.

The tests:

* Four observations on MODIS Terra ocean color, PACE OCI and VIIRS SNPP
  ocean color: three `matched` and one `rejected` at exclusion level 5
  (coefficient of variation above 0.15).
* Two PRISM observations (ORNL DAAC): BioSCape L2A reflectance in False Bay
  and PACE-PAX L1B radiance off California, both `matched`.
* A request without a bounding box, which Harmony must refuse with HTTP 422
  before searching CMR.

Each observation's row is compared with a reference CSV.

## Set the Harmony environment

The cell below sets `harmony_host_url` to one of:

* Production: <https://harmony.earthdata.nasa.gov> (not yet configured: the
  service is not in production)
* UAT: <https://harmony.uat.earthdata.nasa.gov>
* SIT: <https://harmony.sit.earthdata.nasa.gov>
* Local: <http://localhost:3000>

The default is UAT. The cell is tagged `parameters`, so papermill can
override it.

In [ ]:
harmony_host_url = 'https://harmony.uat.earthdata.nasa.gov'

## Prerequisites

The dependencies are in [requirements.txt](./requirements.txt). A `.netrc`
file with Earthdata Login credentials for the target environment must be in
the `test` directory of this repository, or in your home directory when
running the notebook in a browser.

## Reference files

`reference_files` holds the CSV each observation returned in UAT: on
2026-11-06 for the Earthdata Archive observations, on 2026-11-17 for PRISM
and GHRSST (one row each, 38 to 539 columns). The service's
output for a given granule is deterministic, so text columns must match
exactly and numeric columns to a relative tolerance of 1e-6.

### Import required packages

In [ ]:
import datetime as dt
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
from harmony import BBox, Client, Collection, Environment, Request

#### Import shared utility functions

In [ ]:
sys.path.append('../shared_utils')
from utilities import print_error, print_success

### Set up test configuration

Each test is one observation: a collection, a position and a time, with the
status the service must return. SIT uses UAT's CMR, so both use the same
granules. PRISM scenes are a few kilometres across, so their box is ±0.01°
instead of ±0.25°.

In [ ]:
non_production_configuration = [
    {
        'name': 'MODIST_L2_OC, off San Diego',
        'collection_id': 'C1272944705-OB_CLOUD',
        'lat': 33.0,
        'lon': -119.5,
        'time': dt.datetime(2010, 10, 10, 18, 7, 30),
        'status': 'matched',
        'exclusion_levels': '',
        'reference': 'MODIST_L2_OC_san_diego.csv',
    },
    {
        'name': 'PACE_OCI_L2_AOP_NRT, western Mediterranean',
        'collection_id': 'C1281448693-OB_CLOUD',
        'lat': 40.596,
        'lon': 5.813,
        'time': dt.datetime(2026, 6, 3, 11, 35),
        'status': 'matched',
        'exclusion_levels': '',
        'reference': 'PACE_OCI_L2_AOP_NRT_mediterranean.csv',
    },
    {
        'name': 'PACE_OCI_L2_AOP_NRT, Baltic',
        'collection_id': 'C1281448693-OB_CLOUD',
        'lat': 55.914,
        'lon': 19.015,
        'time': dt.datetime(2026, 6, 3, 11, 35),
        'status': 'rejected',
        'exclusion_levels': '5',
        'reference': 'PACE_OCI_L2_AOP_NRT_baltic.csv',
    },
    {
        'name': 'VIIRSN_L2_OC, eastern Black Sea',
        'collection_id': 'C1272968279-OB_CLOUD',
        'lat': 42.925,
        'lon': 40.537,
        'time': dt.datetime(2017, 4, 6, 9, 33),
        'status': 'matched',
        'exclusion_levels': '',
        'reference': 'VIIRSN_L2_OC_black_sea.csv',
    },
    {
        'name': 'BioSCape_PRISM_L2A_RFL (ORNL DAAC), False Bay',
        'collection_id': 'C1295244402-ORNL_CLOUD',
        'lat': -34.30502,
        'lon': 18.58225,
        'time': dt.datetime(2023, 11, 9, 12, 22),
        'half_box_degrees': 0.01,
        'status': 'matched',
        'exclusion_levels': '',
        'reference': 'BioSCape_PRISM_L2A_RFL_false_bay.csv',
    },
    {
        'name': 'PRISM_L1B_PACEPAX_2024 (ORNL DAAC), off California',
        'collection_id': 'C1283055376-ORNL_CLOUD',
        'lat': 37.205,
        'lon': -122.57,
        'time': dt.datetime(2024, 9, 15, 20, 1, 30),
        'half_box_degrees': 0.01,
        'status': 'matched',
        'exclusion_levels': '',
        'reference': 'PRISM_L1B_PACEPAX_california.csv',
    },
]

environment_configuration = {
    # The service is not yet in production; add a production configuration
    # once it is.
    'https://harmony.uat.earthdata.nasa.gov': {
        'config': non_production_configuration,
        'env': Environment.UAT,
    },
    'https://harmony.sit.earthdata.nasa.gov': {
        'config': non_production_configuration,
        'env': Environment.SIT,
    },
    'http://localhost:3000': {
        'config': non_production_configuration,
        'env': Environment.LOCAL,
    },
}

configuration = environment_configuration.get(harmony_host_url)

if configuration is not None:
    harmony_client = Client(env=configuration['env'])
else:
    print(f'satellite-in-situ-matchup is not configured for {harmony_host_url}')

### Helper functions

In [ ]:
def matchup_request(collection_id, lat, lon, time, test_label, half_box_degrees=0.25):
    """The request for one observation: a box and a 1-hour range centered on it.

    Labeled with the suite and the test, so each job can be told apart in the
    Harmony workflow UI.
    """
    half_window = dt.timedelta(minutes=30)
    return Request(
        collection=Collection(id=collection_id),
        spatial=BBox(
            lon - half_box_degrees,
            lat - half_box_degrees,
            lon + half_box_degrees,
            lat + half_box_degrees,
        ),
        temporal={'start': time - half_window, 'stop': time + half_window},
        variables=['all'],
        format='text/csv',
        labels=['satellite-in-situ-matchup-rtest', test_label],
    )


def read_rows(job_id):
    """Wait for a submitted job and return its CSV rows."""
    harmony_client.wait_for_processing(job_id)
    status = harmony_client.status(job_id)
    assert status['status'] == 'successful', f'Job {job_id}: {status}'
    with TemporaryDirectory() as tmp_dir:
        files = [
            future.result()
            for future in harmony_client.download_all(
                job_id, directory=tmp_dir, overwrite=True
            )
        ]
        csvs = sorted(name for name in files if str(name).endswith('.csv'))
        assert len(csvs) == 1, f'Job {job_id}: expected one CSV, got {files}'
        return pd.read_csv(csvs[0], dtype=str, keep_default_na=False)


def as_floats(values):
    """The column as floats, or None if any value is not a number (text, or
    an empty cell)."""
    try:
        return values.astype(float).to_numpy()
    except ValueError:
        return None


def assert_rows_match(rows, reference, relative_tolerance=1e-6):
    """Same columns in the same order; text equal, numbers within tolerance."""
    assert list(rows.columns) == list(reference.columns), (
        f'Columns differ: {len(rows.columns)} returned, '
        f'{len(reference.columns)} in the reference'
    )
    assert len(rows) == len(reference), f'{len(rows)} rows, expected {len(reference)}'
    for column in reference.columns:
        if rows[column].equals(reference[column]):
            continue
        actual, expected = as_floats(rows[column]), as_floats(reference[column])
        assert (
            actual is not None and expected is not None
        ), f'{column}: {rows[column].tolist()} != {reference[column].tolist()}'
        np.testing.assert_allclose(
            actual, expected, rtol=relative_tolerance, equal_nan=True, err_msg=column
        )

### Run the tests

All the requests are submitted first, so Harmony runs them in parallel. Then
each job's row is checked: the status, the exclusion levels, and every
column against the reference CSV. A failed test does not stop the others;
each failure is printed, and the cell fails at the end.

In [ ]:
if configuration is not None:
    submitted = []
    for test in configuration['config']:
        request = matchup_request(
            test['collection_id'],
            test['lat'],
            test['lon'],
            test['time'],
            f"sism-rtest-{Path(test['reference']).stem}",
            half_box_degrees=test.get('half_box_degrees', 0.25),
        )
        submitted.append((test, harmony_client.submit(request)))
        print(f"Submitted {test['name']}")

    failures = []
    for test, job_id in submitted:
        try:
            rows = read_rows(job_id)
            reference = pd.read_csv(
                Path('reference_files') / test['reference'],
                dtype=str,
                keep_default_na=False,
            )
            assert rows['matchup_status'].tolist() == [test['status']], (
                f"status {rows['matchup_status'].tolist()}, "
                f"expected {test['status']}"
            )
            assert rows['exclusion_levels'].tolist() == [test['exclusion_levels']], (
                f"exclusion levels {rows['exclusion_levels'].tolist()}, "
                f"expected {test['exclusion_levels']!r}"
            )
            assert_rows_match(rows, reference)
        except AssertionError as error:
            failures.append(f"{test['name']} (job {job_id}): {error}")
            print_error(failures[-1])
        else:
            print_success(
                f"{test['name']}: {test['status']}, {len(rows.columns)} columns "
                f'as in the reference (job {job_id})'
            )

    summary = f'{len(failures)} of {len(submitted)} tests failed:\n'
    assert not failures, summary + '\n'.join(failures)

### A request without a bounding box

The service needs a bounding box to know where the observation is. Harmony
must refuse such a request with HTTP 422, naming the service, before it
searches CMR or starts a job.

In [ ]:
if configuration is not None:
    test = configuration['config'][0]
    half_window = dt.timedelta(minutes=30)
    incomplete_request = Request(
        collection=Collection(id=test['collection_id']),
        temporal={
            'start': test['time'] - half_window,
            'stop': test['time'] + half_window,
        },
        format='text/csv',
    )
    try:
        harmony_client.submit(incomplete_request)
    except Exception as error:
        assert 'satellite-in-situ-matchup requires spatialSubset' in str(
            error
        ), f'Unexpected error: {error}'
        print_success(f'Request without a bounding box refused: {error}')
    else:
        raise AssertionError('Harmony accepted a request without a bounding box')